# <font color="#418FDE" size="6.5" uppercase>**Support Classification**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Explain maximum-margin classification through practical SVM behavior. 
- Fit linear and kernel SVM classifiers for binary and multiclass tasks. 
- Tune SVM hyperparameters and evaluate class weighting and probability options. 


## **1. SVM Intuition**

### **1.1. Maximum Margin**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_01_01.jpg?v=1783998205" width="250">



>* SVMs choose the widest separating margin
>* Wider margins make predictions more stable

>* Choose the widest gap between classes
>* Nearest points shape the SVM boundary

>* SVMs trade perfect fit for generalization
>* Some errors allow a wider margin



### **1.2. Linear SVC**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_01_02.jpg?v=1783998209" width="250">



>* Straight boundary separates classes
>* Widest margin improves robustness

>* Straight boundaries model approximately linear patterns
>* Margin trade-offs tolerate noisy, overlapping data

>* Strong for sparse, high-feature text data
>* Limited when patterns are strongly nonlinear



In [ ]:
#@title Python Code - Linear SVC

# This script demonstrates a Linear SVC boundary.
# The margin idea appears through support vectors.
# The plot shows separation and borderline points.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import sklearn

# Create a small two-class dataset with visible separation.
features, labels = make_blobs(
    n_samples=120,
    centers=2,
    cluster_std=1.25,
    random_state=42,
)

# Check that the generated data has the expected shape.
if features.shape != (120, 2):
    raise ValueError("Expected 120 rows and 2 features.")

# Split data so evaluation uses examples not seen during training.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Fit a linear support vector classifier with feature scaling.
model = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", C=1.0, random_state=42),
)
model.fit(X_train, y_train)

# Measure how well the learned straight boundary generalizes.
predicted_labels = model.predict(X_test)
accuracy = accuracy_score(y_test, predicted_labels)

# Extract the trained SVC and scaled training data.
scaler = model.named_steps["standardscaler"]
svc = model.named_steps["svc"]
scaled_train = scaler.transform(X_train)

# Build a grid in scaled feature space for the boundary.
x_min = scaled_train[:, 0].min() - 0.8
x_max = scaled_train[:, 0].max() + 0.8
y_min = scaled_train[:, 1].min() - 0.8
y_max = scaled_train[:, 1].max() + 0.8

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250),
)

# Decision scores mark the boundary and margin lines.
grid_points = np.c_[xx.ravel(), yy.ravel()]
decision_scores = svc.decision_function(grid_points).reshape(xx.shape)

print("scikit-learn version:", sklearn.__version__)
print("Test accuracy:", round(accuracy, 3))
print("Support vectors:", len(svc.support_vectors_))

# Plot the learned boundary, margins, and support vectors.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    scaled_train[:, 0],
    scaled_train[:, 1],
    c=y_train,
    cmap="coolwarm",
    s=35,
    label="training examples",
)

ax.contour(
    xx,
    yy,
    decision_scores,
    levels=[-1, 0, 1],
    colors=["gray", "black", "gray"],
    linestyles=["--", "-", "--"],
)

ax.scatter(
    svc.support_vectors_[:, 0],
    svc.support_vectors_[:, 1],
    s=130,
    facecolors="none",
    edgecolors="yellow",
    linewidths=2,
    label="support vectors",
)

ax.set_title("Linear SVC: widest-margin straight boundary")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
ax.legend(loc="best")
plt.show()



### **1.3. Support Vectors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_01_03.jpg?v=1783998207" width="250">



>* Closest points define the SVM margin
>* Moving support vectors can shift the boundary

>* SVMs rely on key borderline examples
>* Ambiguous cases shape the decision boundary

>* Margins ignore safe points, improving robustness
>* Borderline support vectors reveal sensitive cases



In [ ]:
#@title Python Code - Support Vectors

# This script shows which points anchor an SVM.
# Support vectors sit closest to the decision boundary.
# The plot highlights margin-defining training examples.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# Create a small two-class dataset with clear separation.
features, labels = make_blobs(
    n_samples=60,
    centers=2,
    cluster_std=1.15,
    random_state=42,
)

# Validate the simple shape expected by this lesson.
if features.shape != (60, 2):
    raise ValueError("Expected 60 rows and 2 features.")

# Fit a linear SVM so the margin is easy to see.
svm_model = SVC(kernel="linear", C=1.0, random_state=42)
svm_model.fit(features, labels)

# Extract the training points that define the margin.
support_vectors = svm_model.support_vectors_
support_indices = svm_model.support_

# Build a grid for drawing the learned decision regions.
x_min = features[:, 0].min() - 1.0
x_max = features[:, 0].max() + 1.0
y_min = features[:, 1].min() - 1.0
y_max = features[:, 1].max() + 1.0

# Predict the SVM score across the grid.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250),
)
grid_points = np.c_[xx.ravel(), yy.ravel()]
scores = svm_model.decision_function(grid_points).reshape(xx.shape)

# Print concise facts that connect geometry to behavior.
print("scikit-learn version:", sklearn.__version__)
print("Training points:", len(features))
print("Support vectors:", len(support_vectors))
print("Support vector indices:", support_indices.tolist())

# Plot all points, the boundary, margins, and support vectors.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(features[:, 0], features[:, 1], c=labels, cmap="coolwarm", s=45)
ax.contour(xx, yy, scores, levels=[-1, 0, 1], colors=["gray", "black", "gray"])
ax.scatter(
    support_vectors[:, 0],
    support_vectors[:, 1],
    s=180,
    facecolors="none",
    edgecolors="gold",
    linewidths=2,
    label="support vectors",
)

# Label the geometry shown in the figure.
ax.set_title("Linear SVM: support vectors define the margin")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(loc="best")
plt.show()



## **2. Kernel SVC**

### **2.1. Kernel SVC**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_02_01.jpg?v=1783998199" width="250">



>* Handles curved, nonlinear class boundaries
>* Uses kernels for flexible maximum-margin separation

>* Choose kernels based on boundary shape
>* Validate flexible kernels to avoid overfitting

>* Scale features and tune kernel settings
>* Supports binary and multiclass classification



In [ ]:
#@title Python Code - Kernel SVC

# Compare linear and kernel SVC boundaries.
# Scaling supports distance based kernel behavior.
# The plot shows nonlinear separation clearly.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Create a small nonlinear binary classification dataset.
features, labels = make_circles(
    n_samples=300,
    factor=0.45,
    noise=0.08,
    random_state=42,
)

# Check that the generated data has the expected shape.
if features.shape != (300, 2):
    raise ValueError("Expected 300 rows and 2 features.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42,
)

# Fit one linear SVC and one RBF kernel SVC.
linear_model = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0))
rbf_model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale"))
linear_model.fit(X_train, y_train)
rbf_model.fit(X_train, y_train)

# Compare test accuracy for the two boundaries.
linear_accuracy = accuracy_score(y_test, linear_model.predict(X_test))
rbf_accuracy = accuracy_score(y_test, rbf_model.predict(X_test))
print("scikit-learn version:", sklearn.__version__)
print("Linear SVC test accuracy:", round(linear_accuracy, 3))
print("RBF kernel SVC test accuracy:", round(rbf_accuracy, 3))

# Build a grid to visualize the RBF decision regions.
x_min = features[:, 0].min() - 0.3
x_max = features[:, 0].max() + 0.3
y_min = features[:, 1].min() - 0.3
y_max = features[:, 1].max() + 0.3

# Predict the class for each grid location.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 180),
    np.linspace(y_min, y_max, 180),
)
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_predictions = rbf_model.predict(grid_points).reshape(xx.shape)

# Plot the nonlinear boundary learned by the kernel SVC.
fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, grid_predictions, alpha=0.25, cmap="coolwarm")
ax.scatter(features[:, 0], features[:, 1], c=labels, cmap="coolwarm", s=28)
ax.set_title("RBF Kernel SVC Learns a Curved Boundary")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
plt.show()



### **2.2. NuSVC**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_02_02.jpg?v=1783998201" width="250">



>* Nu controls margin errors and support vectors
>* Kernel NuSVC handles noisy nonlinear boundaries

>* Choose kernels to match boundary shape
>* Tune nu to avoid overfitting or underfitting

>* Binary decisions combine for multiclass NuSVC
>* Tune nu carefully for imbalanced classes



In [ ]:
#@title Python Code - NuSVC

# This example fits a kernel NuSVC classifier.
# It shows preprocessing before nonlinear SVM training.
# The output compares accuracy and support vectors.

import sklearn
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import NuSVC

# Create a small nonlinear binary classification dataset.
features, labels = make_moons(n_samples=300, noise=0.25, random_state=42)

# Check that the feature matrix and labels match.
if features.shape[0] != labels.shape[0]:
    raise ValueError("Features and labels must have matching rows.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# StandardScaler is fitted only on training data inside the pipeline.
model = make_pipeline(
    StandardScaler(), NuSVC(kernel="rbf", nu=0.35, gamma="scale")
)

# Fit one nonlinear NuSVC decision boundary.
model.fit(X_train, y_train)

# Evaluate predictions on the held-out test set.
predicted_labels = model.predict(X_test)
accuracy = accuracy_score(y_test, predicted_labels)

# Count support vectors from the fitted NuSVC step.
svc_step = model.named_steps["nusvc"]
support_vector_count = int(svc_step.n_support_.sum())

print("scikit-learn version:", sklearn.__version__)
print("Test accuracy:", round(accuracy, 3))
print("Support vectors used:", support_vector_count)



### **2.3. Multiclass SVM Strategies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_02_03.jpg?v=1783998203" width="250">



>* Multiclass SVMs combine binary decisions
>* Kernel boundaries handle complex class patterns

>* Train classifiers for every class pair
>* Combine pairwise votes for final prediction

>* One-versus-rest trains one classifier per class
>* Strategy affects cost, interpretation, and performance



In [ ]:
#@title Python Code - Multiclass SVM Strategies

# This example compares multiclass SVM strategies.
# Pairwise and rest strategies combine binary margins.
# Accuracy and classifier counts reveal practical differences.

import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Load a small three-class dataset for multiclass classification.
iris = load_iris()
X = iris.data
y = iris.target

# Check that the lesson really has more than two classes.
class_count = len(set(y))
if class_count < 3:
    raise ValueError("This example needs at least three classes.")

# Split data with stratification to preserve class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Use the same kernel SVC inside each multiclass strategy.
base_svc = SVC(kernel="rbf", C=1.0, gamma="scale")
ovo_model = make_pipeline(StandardScaler(), OneVsOneClassifier(base_svc))
ovr_model = make_pipeline(StandardScaler(), OneVsRestClassifier(base_svc))

# Fit both strategies on the same training data.
ovo_model.fit(X_train, y_train)
ovr_model.fit(X_train, y_train)

# Compare predictions on the same held-out test data.
ovo_accuracy = accuracy_score(y_test, ovo_model.predict(X_test))
ovr_accuracy = accuracy_score(y_test, ovr_model.predict(X_test))

# Count how many binary SVMs each strategy trained.
ovo_count = len(ovo_model.named_steps["onevsoneclassifier"].estimators_)
ovr_count = len(ovr_model.named_steps["onevsrestclassifier"].estimators_)

print("scikit-learn version:", sklearn.__version__)
print("Classes:", class_count)
print("One-versus-one binary SVMs:", ovo_count)
print("One-versus-rest binary SVMs:", ovr_count)
print("One-versus-one test accuracy:", round(ovo_accuracy, 3))
print("One-versus-rest test accuracy:", round(ovr_accuracy, 3))



## **3. SVC Tuning**

### **3.1. Probability Estimates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_03_01.jpg?v=1783998210" width="250">



>* SVC margins are not probabilities
>* Probabilities support clearer real-world decisions

>* Calibration adds cost and changes model behavior
>* Check probabilities for real-world reliability

>* Use probabilities to tune decision thresholds
>* Enable only when calibrated estimates add value



In [ ]:
#@title Python Code - Probability Estimates

# This example compares SVC probability estimates.
# Calibration turns margins into estimated probabilities.
# The output shows accuracy and probability quality.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Create a small binary dataset with overlapping classes.
features, target = make_classification(
    n_samples=600,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    class_sep=0.9,
    random_state=42,
)

# Split data so evaluation uses examples unseen during training.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.30,
    stratify=target,
    random_state=42,
)

# Validate the expected binary classification setup.
if len(np.unique(y_train)) != 2:
    raise ValueError("This lesson needs exactly two classes.")

# Enable probability estimates for the support vector classifier.
svc_model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42),
)

# Fit once, then compare labels, margins, and probabilities.
svc_model.fit(X_train, y_train)
predicted_labels = svc_model.predict(X_test)
positive_probabilities = svc_model.predict_proba(X_test)[:, 1]
decision_scores = svc_model.decision_function(X_test)

# Accuracy checks labels, while Brier score checks probabilities.
accuracy = accuracy_score(y_test, predicted_labels)
brier = brier_score_loss(y_test, positive_probabilities)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Test accuracy from class labels: {accuracy:.3f}")
print(f"Brier score for probabilities: {brier:.3f}")
print("Lower Brier score means probabilities match outcomes better.")

# Show a few cases near the boundary, where uncertainty is clearest.
near_boundary = np.argsort(np.abs(decision_scores))[:5]
print("Near-boundary class-1 probabilities:")
print(np.round(positive_probabilities[near_boundary], 3))



### **3.2. Class Weighting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_03_02.jpg?v=1783998212" width="250">



>* Imbalanced data can hide poor minority detection
>* Class weights shift margins toward fairer recognition

>* Match weights to application priorities
>* Tune using metrics and error costs

>* Check per-class metrics, not accuracy alone
>* Tune weights with validation and hyperparameters



In [ ]:
#@title Python Code - Class Weighting

# This example compares SVC class weighting.
# Imbalanced data makes accuracy potentially misleading.
# Recall changes show the weighting tradeoff.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Create a small imbalanced binary classification dataset.
features, target = make_classification(
    n_samples=800,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    weights=[0.9, 0.1],
    class_sep=0.9,
    random_state=42,
)

# Check that both classes are present before splitting.
class_counts = np.bincount(target)
if len(class_counts) != 2 or min(class_counts) == 0:
    raise ValueError("Expected two non-empty classes.")

# Use stratification to preserve the imbalance in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Fit the same SVC once without weights and once with balanced weights.
models = {
    "plain SVC": SVC(kernel="rbf", C=1.0, gamma="scale"),
    "balanced SVC": SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"),
}

# Evaluate metrics that reveal minority-class behavior.
print("scikit-learn version:", sklearn.__version__)
print("test class counts [majority, minority]:", np.bincount(y_test).tolist())
print("model | accuracy | balanced accuracy | minority recall")

for model_name, svc_model in models.items():
    pipeline = make_pipeline(StandardScaler(), svc_model)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    balanced = balanced_accuracy_score(y_test, predictions)
    minority_recall = recall_score(y_test, predictions, pos_label=1)
    print(
        model_name,
        "|",
        round(accuracy, 3),
        "|",
        round(balanced, 3),
        "|",
        round(minority_recall, 3),
    )



### **3.3. C Gamma Tuning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_12/Lecture_A/image_03_03.jpg?v=1783998214" width="250">



>* C balances margin width and training errors
>* Moderate C can improve noisy-data generalization

>* Low gamma creates smoother, broader boundaries
>* High gamma captures detail but risks overfitting

>* Tune C and gamma together
>* Validate using task-appropriate performance metrics



In [ ]:
#@title Python Code - C Gamma Tuning

# Tune C and gamma for an SVC.
# Compare validation accuracy across small settings.
# Identify underfitting and overfitting patterns.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Create a small nonlinear binary classification dataset.
features, labels = make_moons(n_samples=400, noise=0.25, random_state=42)

# Split data so tuning is judged on unseen examples.
X_train, X_valid, y_train, y_valid = train_test_split(
    features, labels, test_size=0.35, stratify=labels, random_state=42
)

# Validate the basic shape before fitting models.
if X_train.shape[1] != 2 or len(np.unique(labels)) != 2:
    raise ValueError("Expected two features and two classes.")

# Try a small grid of C and gamma values.
c_values = [0.1, 1, 10]
gamma_values = [0.1, 1, 10]
accuracy_grid = np.zeros((len(gamma_values), len(c_values)))

# Fit one SVC for each hyperparameter pair.
for row, gamma_value in enumerate(gamma_values):
    for col, c_value in enumerate(c_values):
        model = make_pipeline(
            StandardScaler(), SVC(C=c_value, gamma=gamma_value, kernel="rbf")
        )
        model.fit(X_train, y_train)
        predictions = model.predict(X_valid)
        accuracy_grid[row, col] = accuracy_score(y_valid, predictions)

# Find the best validation result from the grid.
best_index = np.unravel_index(np.argmax(accuracy_grid), accuracy_grid.shape)
best_gamma = gamma_values[best_index[0]]
best_c = c_values[best_index[1]]
best_accuracy = accuracy_grid[best_index]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Best validation accuracy: {best_accuracy:.3f}")
print(f"Best pair: C={best_c}, gamma={best_gamma}")

# Plot validation accuracy for each C and gamma pair.
fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(accuracy_grid, vmin=0.75, vmax=1.0, cmap="viridis")

ax.set_title("SVC validation accuracy for C and gamma")
ax.set_xlabel("C value")
ax.set_ylabel("gamma value")
ax.set_xticks(range(len(c_values)), labels=[str(value) for value in c_values])
ax.set_yticks(range(len(gamma_values)), labels=[str(value) for value in gamma_values])

# Add readable accuracy labels inside the heatmap.
for row in range(len(gamma_values)):
    for col in range(len(c_values)):
        ax.text(col, row, f"{accuracy_grid[row, col]:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax, label="validation accuracy")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Support Classification**</font>


In this lecture, you learned to:
- Explain maximum-margin classification through practical SVM behavior. 
- Fit linear and kernel SVM classifiers for binary and multiclass tasks. 
- Tune SVM hyperparameters and evaluate class weighting and probability options. 

In the next Lecture (Lecture B), we will go over 'SVR Kernel Ridge'